In [2]:
install.packages("factoextra")

also installing the dependencies ‘crosstalk’, ‘DT’, ‘ellipse’, ‘flashClust’, ‘multcompView’, ‘dendextend’, ‘FactoMineR’


Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [3]:
library(tidyverse)
library(factoextra)  # For PCA visualization and k-means
library(repr)

Welcome! Want to learn more? See two factoextra-related books at https://goo.gl/ve3WBa



In [4]:
options(repr.plot.width=8, repr.plot.height=6, repr.plot.res = 300)
library(showtext)
font_add_google("Lato", "lato")
showtext_opts(dpi = 300)
showtext_auto()

Loading required package: sysfonts

Loading required package: showtextdb



In [5]:
# Cut distributions at zero variability point
cut_degree_distributions <- function(node_df) {
  # For table with columns: PID, group, node_id, node_degree
  
  # Find the max degree that has non-zero values across subjects
  max_relevant_degree <- node_df %>%
    group_by(node_id) %>%
    summarize(has_values = any(node_degree > 0)) %>%
    filter(has_values == TRUE) %>%
    pull(node_id) %>%
    max()
  
  # Filter the dataframe to include only nodes up to the max relevant degree
  node_df %>%
    filter(node_id <= max_relevant_degree)
}

# Perform PCA and find knee point in eigenspectrum
perform_pca_with_knee <- function(data, id_col = 1, method = "percentvar", threshold = 0.9) {
  # Remove ID column for PCA
  pca_data <- data %>% select(-all_of(id_col))
  
  # Run PCA
  pca_result <- prcomp(pca_data, scale. = TRUE)
  
  # Find knee point in eigenspectrum using factoextra
  if (method == "percentvar") {
    # Keep components that explain threshold% of variance
    var_explained <- pca_result$sdev^2 / sum(pca_result$sdev^2)
    cum_var <- cumsum(var_explained)
    knee_point <- which(cum_var >= threshold)[1]
  } else {
    # Use factoextra's get_eigenvalue for a more sophisticated approach
    eigenvalues <- factoextra::get_eigenvalue(pca_result)
    knee_point <- factoextra::fviz_eig(pca_result, choice = "eigenvalue", 
                                       addlabels = FALSE, ncp = 20)$data %>%
      arrange(desc(diff)) %>%
      pull(dim) %>%
      first()
  }
  
  # Return PCA results, optimal dimensions, and data for plotting
  list(
    pca_obj = pca_result,
    knee_point = knee_point,
    pca_data = as_tibble(pca_result$x) %>% 
      bind_cols(data %>% select(all_of(id_col))),
    var_explained = var_explained
  )
}

# Plot eigenspectrum with knee point
plot_eigenspectrum <- function(pca_result) {
  var_explained <- pca_result$var_explained
  knee_point <- pca_result$knee_point
  
  tibble(
    component = 1:length(var_explained),
    variance = var_explained,
    cumulative = cumsum(var_explained)
  ) %>%
    pivot_longer(cols = c(variance, cumulative), 
                 names_to = "type", 
                 values_to = "value") %>%
    ggplot(aes(x = component, y = value, color = type)) +
    geom_line() +
    geom_point() +
    geom_vline(xintercept = knee_point, linetype = "dashed", color = "red") +
    annotate("text", x = knee_point + 1, y = max(var_explained), 
             label = paste("Knee =", knee_point), hjust = 0) +
    scale_x_continuous(breaks = 1:length(var_explained)) +
    labs(title = "PCA Eigenspectrum with Knee Point",
         x = "Principal Component",
         y = "Proportion of Variance") +
    theme_minimal()
}

# Visualize first 2 PCs
plot_first_two_pcs <- function(pca_result, id_col_name = "subject_id") {
  pca_result$pca_data %>%
    ggplot(aes(x = PC1, y = PC2)) +
    geom_point(alpha = 0.7) +
    stat_ellipse(level = 0.95, linetype = 2) +
    labs(title = "Distribution of First Two Principal Components",
         x = paste0("PC1 (", round(pca_result$var_explained[1] * 100, 1), "%)"),
         y = paste0("PC2 (", round(pca_result$var_explained[2] * 100, 1), "%)")) +
    theme_minimal()
}

# Whiten data and perform k-means clustering
cluster_data <- function(pca_result, k = 2, pcs_to_use = NULL) {
  if (is.null(pcs_to_use)) {
    pcs_to_use <- 1:pca_result$knee_point
  }
  
  # Extract PCs to use and whiten (scale to unit variance)
  whitened_data <- pca_result$pca_data %>%
    select(all_of(paste0("PC", pcs_to_use))) %>%
    scale()
  
  # Perform k-means clustering
  km_result <- kmeans(whitened_data, centers = k, nstart = 25)
  
  # Add cluster assignments to original PCA data
  clustered_data <- pca_result$pca_data %>%
    mutate(cluster = as.factor(km_result$cluster))
  
  # Return all results
  list(
    km_obj = km_result,
    clustered_data = clustered_data,
    whitened_data = whitened_data
  )
}

# Visualize clusters in PC space
plot_clusters <- function(cluster_result) {
  cluster_result$clustered_data %>%
    ggplot(aes(x = PC1, y = PC2, color = cluster)) +
    geom_point(alpha = 0.7) +
    stat_ellipse(aes(group = cluster), level = 0.95) +
    labs(title = paste("K-means Clustering (k =", 
                       length(unique(cluster_result$clustered_data$cluster)), ")"),
         x = "PC1", y = "PC2") +
    theme_minimal() +
    scale_color_brewer(palette = "Set1")
}

# Analyze distribution of groups between clusters
analyze_cluster_groups <- function(cluster_result, group_col) {
  cluster_result$clustered_data %>%
    count(cluster, !!sym(group_col)) %>%
    group_by(cluster) %>%
    mutate(proportion = n / sum(n)) %>%
    ungroup() %>%
    ggplot(aes(x = cluster, y = proportion, fill = !!sym(group_col))) +
    geom_col() +
    labs(title = "Distribution of Groups Between Clusters",
         x = "Cluster", y = "Proportion") +
    theme_minimal() +
    scale_y_continuous(labels = scales::percent)
}

# Full workflow function that runs the complete pipeline
run_clustering_workflow <- function(node_degree_df, 
                                   id_col = 1, 
                                   group_col = NULL,
                                   pca_threshold = 0.9,
                                   k = 2) {
  # Step 1: Cut distributions at zero variability
  cut_data <- cut_degree_distributions(node_degree_df)
  
  # Step 2: Perform PCA and find knee point
  pca_result <- perform_pca_with_knee(cut_data, id_col, threshold = pca_threshold)
  
  # Step 3: Plot eigenspectrum
  eigen_plot <- plot_eigenspectrum(pca_result)
  
  # Step 4: Visualize first 2 PCs
  pc_plot <- plot_first_two_pcs(pca_result)
  
  # Step 5: Whiten and cluster
  cluster_result <- cluster_data(pca_result, k = k)
  
  # Step 6: Plot clusters
  cluster_plot <- plot_clusters(cluster_result)
  
  # Step 7: Analyze group distribution if a group column is provided
  group_analysis <- NULL
  if (!is.null(group_col)) {
    group_analysis <- analyze_cluster_groups(cluster_result, group_col)
  }
  
  # Return all results
  list(
    cut_data = cut_data,
    pca_result = pca_result,
    eigen_plot = eigen_plot,
    pc_plot = pc_plot,
    cluster_result = cluster_result,
    cluster_plot = cluster_plot,
    group_analysis = group_analysis
  )
}